# 07 - Analyse de Performance du SAD

Objectif phase 7:
- Taux de couverture automatique
- Accuracy par tranche de confiance
- Verification de l objectif accuracy > 95% pour confiance > 0.85
- Analyse cout-benefice: Cost_total = (FN x 1000) + (FP x 100) + (Revision x 50)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.decision.engine import generer_recommandation
from src.decision.rules import appliquer_regle_securite_negatif
from src.decision.triage import appliquer_triage
from src.evaluation.analysis import analyser_performance_sad, resume_business_markdown

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
RNG = np.random.default_rng(2026)
print("Imports OK")

Imports OK


## 1) Simulation d un lot de predications

In [2]:
n_cases = 120
probas = RNG.dirichlet(alpha=[1.4, 1.2, 1.8, 1.1], size=n_cases)

# Simuler quelques faux negatifs potentiels (pred notumor confiante)
for idx in [7, 33, 89]:
    probas[idx] = np.array([0.02, 0.03, 0.92, 0.03])

patient_ids = [f"P_{i+1:05d}" for i in range(n_cases)]
probas[:2]

array([[0.13012702, 0.00849134, 0.70597234, 0.1554093 ],
       [0.1678937 , 0.35598546, 0.28697771, 0.18914313]])

## 2) Pipeline decisionnel SAD

In [3]:
decisions = []
for pid, p in zip(patient_ids, probas):
    d = generer_recommandation(p, CLASSES, patient_id=pid)
    d = appliquer_regle_securite_negatif(d)
    d = appliquer_triage(d)
    decisions.append(d)

len(decisions)

120

## 3) Construction du jeu d evaluation

In [4]:
# Simuler y_true pour l evaluation metier (demo).
y_true = RNG.choice(CLASSES, size=n_cases, p=[0.28, 0.22, 0.30, 0.20])
y_pred = [d.classe_predite for d in decisions]
confiances = [d.confiance for d in decisions]
revision_requise = [d.revision_requise for d in decisions]

df_eval = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "confiance": confiances,
    "revision_requise": revision_requise,
})
df_eval.head()

,y_true,y_pred,confiance,revision_requise
0,meningioma,notumor,0.705972,True
1,pituitary,meningioma,0.355985,True
2,glioma,notumor,0.321016,True
3,notumor,glioma,0.672639,True
4,notumor,meningioma,0.534758,True


## 4) Metriques metier phase 7

In [5]:
resultats = analyser_performance_sad(
    y_true=y_true,
    y_pred=y_pred,
    confiances=confiances,
    revision_requise=revision_requise,
)

print(resume_business_markdown(resultats))

### Resume metier SAD
- Accuracy globale: 21.67%
- Couverture automatique: 0.00%
- Accuracy confiance > 0.85: 0.00% (objectif > 95%)
- FN: 39
- FP: 21
- Revisions: 120
- Cost_total: 47100


In [6]:
print("Accuracy par tranche de confiance")
display(resultats["accuracy_par_tranche"])

obj = resultats["objectif_haute_confiance"]
print("")
print("Objectif haute confiance")
print("- n cas:", obj["n_cas"])
print("- accuracy:", obj["accuracy"])
print("- objectif > 0.95 atteint:", obj["objectif_atteint"])

print("")
print("Couts")
print(resultats["couts"])

Accuracy par tranche de confiance


,tranche,n_cas,accuracy
0,"[0.00, 0.50[",74,0.202703
1,"[0.50, 0.65[",29,0.275862
2,"[0.65, 0.85[",14,0.214286
3,"[0.85, 1.00[",3,0.000000



Objectif haute confiance
- n cas: 3
- accuracy: 0.0
- objectif > 0.95 atteint: False

Couts
{'FN': 39, 'FP': 21, 'Revision': 120, 'Cost_total': 47100}
